# Step 4 — HTML → Markdown Parser

At this stage we **do not perform crawling again**.

We will:

1. Read HTML from `pages` table.
2. Extract main content using **trafilatura**.
3. Save results as Markdown to `markdown` column.
4. Verify parsing results.

In [6]:
import warnings
warnings.filterwarnings('ignore')

In [7]:
from pathlib import Path
import sqlite3
import trafilatura

DB_PATH = Path("../data/linux_docs.db")
assert DB_PATH.exists(), "Database not found. Run Step 2 and Step 3 first."

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

## Fetch pages that haven't been parsed

In [8]:
cursor.execute("""
SELECT id, url, html
FROM pages
WHERE markdown IS NULL OR markdown = ''
ORDER BY id;
""")

rows = cursor.fetchall()
print(f"Number of pages to parse: {len(rows)}")

Number of pages to parse: 0


## Parse HTML to Markdown

In [9]:
updated = 0

for row in rows:
    markdown = trafilatura.extract(
        row["html"],
        output_format="markdown",
        include_links=True,
        include_images=False,
        favor_precision=True,
    )

    if markdown:
        cursor.execute(
            '''
            UPDATE pages
            SET markdown = ?
            WHERE id = ?
            ''',
            (markdown, row["id"])
        )
        updated += 1

conn.commit()

print(f"✅ Successfully updated {updated} pages.")

✅ Successfully updated 0 pages.


## View results

In [10]:
cursor.execute("""
SELECT id, url, title, markdown
FROM pages
ORDER BY id DESC
LIMIT 1;
""")

page = cursor.fetchone()

print("Title:", page["title"])
print("URL  :", page["url"])
print("-" * 80)
print(page["markdown"][:2000])

Title: Installation guide - ArchWiki
URL  : https://wiki.archlinux.org/title/Installation_guide
--------------------------------------------------------------------------------
This document is a guide for installing [Arch Linux](/title/Arch_Linux) using the live system booted from an installation medium made from an official installation image. Its accessibility features are described on the page [Install Arch Linux with accessibility options](/title/Install_Arch_Linux_with_accessibility_options). For alternative means of installation, see [Category:Installation process](/title/Category:Installation_process).

Before installing, it would be advised to view the [FAQ](/title/FAQ). For conventions used in this document, see [Help:Reading](/title/Help:Reading). In particular, code examples may contain placeholders (formatted in *italics*

This guide is kept concise and you are advised to follow the instructions in the presented order per section. For more detailed instructions, see the re

## Notes

At the end of Step 4:

- Raw HTML is still preserved.
- Extracted Markdown is also preserved.
- We don't need to crawl again if we want to change chunking strategy.

Next we will use the `markdown` column as input for the chunking process.

In [11]:
conn.close()
print("Done ✅")

Done ✅
